Component Libraries

In [20]:
import numpy as np
import pandas as pd

**NOTE:**

**What is my goal:** 

My goal is to roughly estimate the timing diagram or bits so that i can match it after I code in verilog.

In verilog if that module has a specification that the input number is treated as suppose Q3.2 format then the input to the module 01001 will be 2.25 in decimal. But in python to get the binary representation and mimic the computation you need to perform computations using 
2.25 * (1 << 2) which is 2.25 * (2 power 2).



In [21]:
#USe format instead of bin(5)
x=8
print(format(-5,f"0{x}b"))

-0000101


**Elementary functions only in python**

I have turned off the bit growth now to simulate realistic scenario. If you want accurate results just ensure you have bits in your arsenal.

These functions are just written to convert from binary strings and to plot and check in python

In [22]:
def binary_to_decimal(signed_binary_str: str)-> int:

    if(signed_binary_str.startswith("0b")):
        signed_binary_str= signed_binary_str[2:]

    return (int(signed_binary_str[1:],2) - (1<< len(signed_binary_str)-1)) if signed_binary_str[0] == '1' else int(signed_binary_str,2)

def twos_complement_negate(signed_binary_str: str) -> str:
    # still 1st bit is signed.
    return format((1 << len(signed_binary_str)) - binary_to_decimal('0' + signed_binary_str), f"0{len(signed_binary_str)}b")

def truncate(binary_str: str , tot_bits: int):
    if len(binary_str)<tot_bits:
        raise ValueError(f"Invalid value for tot_bits {tot_bits}: must be less than the length of binary string {binary_str}")
    return binary_str[(len(binary_str)-tot_bits):]

def decimal_to_signed_binary(decimal_num : int, tot_bit: int) -> str:

    if decimal_num>=0:
        return truncate('0'+format(decimal_num, f"0{tot_bit-1}b"), tot_bit)
    else:
        return truncate(twos_complement_negate('0' + format(decimal_num, f"0{tot_bit}b")[1:]),tot_bit)





#---------------------------fractions---------------------------
# def float_to_fixedstr(signed_float_num: float,frac_bits:int) -> str:
#     return decimal_to_signed_binary(int(signed_float_num*(1<<frac_bits)), )

    

**VERILOG BLOCKS:**

**1. The rest functions are verilog blocks , may be inbuilt or self made. Trick is that don't interconvert bases here just work with binary strings.**

**2. Use loops only when next iteration don't depend on previous iteration ( just reducing the tediousness by not writing number of times )**

**3. Or when there is a inbuilt block exists and perform better. You just care about the output here.**

**4. Using loops dont always implies a sequential logic. It could be combinational logic, as you could use integer i=3; for(...) in verilog. It just creates multiple hardwares**

In [ ]:
'''
--------------------------------------------------------------------------Above this will just be used by python-------------------------

Meaning here the conversions are not used in verilog, everything is binary there. Just here I need to plot and all thats why.
binary_add has a better implementation in hardware so i dont need to write from scratch.

1 << shift has a simple hardware implementation but writing the arithmetic_left_shift again as its implemented as i dont want to use decimal numbers beyond this point.
In verilog bin num << shift (int) is used so its useful and easier here if I leave shift as int here
'''

#Adds 2 signed two complement numbers (I have a better block in verilog)
# def binary_add(A: str, B :str ) -> str :
#      return decimal_to_signed_binary(binary_to_decimal(A) + binary_to_decimal(B), max(len(A),len(B)))


'''I need to test the below blocks and model it in verilog . So get the rough testbench and timing diagram to compare in verilog.'''

# Now all from scratch what really happens in fpga
def arithmetic_left_shift(bin_str : str, shift: int) -> str:
    return bin_str[shift:]+ '0'*shift

def arithmetic_right_shift(bin_str : str, shift: int) -> str:
    return  '0'*shift + bin_str[:len(bin_str)-shift]

def calculate_shifts(bin_str: str):
    
    # b is unsigned, meaning it will just calculate the shifts not care about how it will be used
    shifts_arr = []
    shift = len(bin_str)-1
    for x in bin_str:
        if x=='1':
            shifts_arr.append(shift)
        shift -=1
    return shifts_arr

def shift_and_add(unsigned_bin_str: str ,shifts_arr: list):

    result= '0'*(len(unsigned_bin_str)+1)
    for x in shifts_arr:
        result = binary_add(result,'0'+arithmetic_left_shift(unsigned_bin_str,x))

    return result[1:]


# Multiplying by shifting and adding logic
def multiply(signed_bin_str1: str, signed_bin_str2: str):
    
    unsigned_bin_str1 = twos_complement_negate(signed_bin_str1)[1:] if signed_bin_str1[0]=='1' else signed_bin_str1[1:]
    unsigned_bin_str2 = twos_complement_negate(signed_bin_str2)[1:] if signed_bin_str2[0]=='1' else signed_bin_str2[1:]

    #-------------------I can combine these 2 blocks in verilog----------------------
    shifts_arr = calculate_shifts(unsigned_bin_str2) # ignoring sign for now

    unsigned_result = shift_and_add(unsigned_bin_str1, shifts_arr)
    #-------------------------------------------------------------------------------

    if signed_bin_str1[0] != signed_bin_str2[0]:
        return twos_complement_negate('0'+ unsigned_result)
    else:
        return '0' + unsigned_result


def unsigned_binary_add(bin_str1, bin_str2):
    return format(int(bin_str1,2) + int(bin_str2,2),f'0{max(len(bin_str1),len(bin_str2))}b')[-max(len(bin_str1),len(bin_str2)):]

def is_leq_unsigned(str1,str2):
    return '1' if int(str1,2)<= int(str2,2) else '0'

def square_root(unsigned_bin_str : str) -> str:

    result='0'*(len(unsigned_bin_str)+1) # A bit growth
    a= ('0'* (len(result)-(1+int(len(unsigned_bin_str)/2)))) + '1' + ('0'*int(len(unsigned_bin_str)/2)) 

    '''The logic of assignment of 'a' is in the copy. Note it down'''

    while a!= '0'*(len(unsigned_bin_str)+1):
        
        num_sq = multiply('0'+unsigned_binary_add(result,a),'0'+unsigned_binary_add(result,a))[1:]
        
        if is_leq_unsigned(num_sq,'0'+unsigned_bin_str)=='1':
            result = unsigned_binary_add(result,a)

        a = arithmetic_right_shift(a,1)

    return result[1:]

**TESTS**

In [52]:
string1 = "00100"
string2 = "00011"
print(f'{string1} : {binary_to_decimal(string1)}\n {string2} : {binary_to_decimal(string2)}')
# print(8<<[1,2])
print(f'Multiplication : {multiply(string1,string2)} -> {binary_to_decimal(multiply(string1,string2))}')


00100 : 4
 00011 : 3
Multiplication : 01100 -> 12


In [75]:
# print(is_leq_unsigned('1101','1110'))
N='0001001'
# a='1' + '0'*(8-1)
# print(multiply('0'+a,'0'+a))
print(square_root(N))

0000011


Dont fear to make mistakes. If there will be a problem in future i will deal in the future . Deal with best of your capabilities for the current problem. After many reapeatations you will avoid future problems.